In [ ]:
import numpy as np
import pandas as pd
from config import *

In [ ]:
fix_partial_reporting=False
fix_outliers=False
df_hosp = read_hosp_incidence_data(data_dir, epiyear, epiweek, states, 
                                   new_format=True, download=False,
                                   fix_partial_reporting=fix_partial_reporting, 
                                   fix_outliers=fix_outliers,
                                   plot=False)
df_hosp = df_hosp[df_hosp.date<pd.to_datetime(ref_date)]


In [ ]:
ensemble_models = ["ExponentialSmoothing_mod_ili_nowcast", 
                   "LightGBM_mod_ili_nowcast", , 
                   "seasonal_drift_mod_ili_nowcast", 
                   "SIR-EAKF_start_251004_nowcast"]
                #    "CU-ARNB_Net"]#,"SIR-EAKF_start_250906"],"SIR-EAKF_start_250809"]

mean_ensemble_name = 'Mean_CU_Ensemble4'
weighted_ensemble_name = 'WIS_Weighted_CU_Ensemble4'

start_date = pd.to_datetime('2025-11-08', format="%Y-%m-%d")

In [ ]:
ensemble_dict = {}
# load models predictions
for model in ensemble_models:
    print("-----------Loading model: {}-----------".format(model))
    df_results = load_pred_result_files(results_dir, model, locations)
    ensemble_dict[model] = df_results

print("-----------Generating mean ensemble: {}-----------".format(mean_ensemble_name))
generate_mean_ensemble_pred_results(ensemble_dict, start_date, results_dir, mean_ensemble_name)
df_metrics_models = calc_models_pred_fit(df_hosp, ensemble_models, season, locations, 
                                  results_dir, figures_dir, alpha_vals, plot=False)
df_weights = generate_pred_weights(ensemble_dict, df_metrics_models, locations, decay_rate=0.5)
print("-----------Generating weighted ensemble: {}-----------".format(weighted_ensemble_name))
generate_weighted_pred_results(ensemble_dict, start_date, df_weights, locations, results_dir, weighted_ensemble_name) 

df_metrics_ensembles = calc_models_pred_fit(df_hosp, [mean_ensemble_name,weighted_ensemble_name],season,
                                            locations, results_dir, figures_dir, alpha_vals, plot=False)

df_metrics = pd.concat([df_metrics_models, df_metrics_ensembles], ignore_index=True)
df_metrics.to_csv(results_dir+"/_metrics/metrics_" +ref_date.strftime("%Y-%m-%d") +'.csv',index=False)

In [ ]:
# df_metrics = pd.read_csv(results_dir+"/_metrics/metrics_" +ref_date.strftime("%Y-%m-%d") +'.csv')
# additional_model = ["FluSight-baseline"] 
# df_metrics = df_metrics[~df_metrics['model'].isin(additional_model)]
# df_metrics_additional = calc_models_pred_fit(df_hosp, additional_model, season, locations, 
#                                              results_dir, figures_dir, alpha_vals, plot=False)
# df_metrics = pd.concat([df_metrics, df_metrics_additional], ignore_index=True) 
# df_metrics.to_csv(results_dir+"/_metrics/metrics_" +ref_date.strftime("%Y-%m-%d") +'.csv',index=False)

In [ ]:
# # models_to_plot = ensemble_models + [mean_ensemble_name,weighted_ensemble_name]
# models_to_plot = [mean_ensemble_name,weighted_ensemble_name] 
# # models_to_plot = "LightGBM", "LightGBM_mod_ili"] 
# # models_to_plot = ["SIR-EAKF_start_250906"]
# calc_models_pred_fit(df_hosp, models_to_plot, season, locations, 
#                      results_dir, figures_dir, alpha_vals, plot=True)